In [1]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient , command , Input , Output
from azure.ai.ml.entities import Data , Environment , Model , ComputeInstance , AmlCompute
from azure.ai.ml.constants import AssetTypes

from datetime import datetime
from zoneinfo import ZoneInfo


from azure.ai.ml.dsl import pipeline


In [2]:
from azure.ai.ml import MLClient
from azure.ai.ml.dsl import pipeline
from azure.ai.ml.entities import (
    BatchEndpoint,
    BatchDeployment,
    BatchRetrySettings
)
from azure.identity import DefaultAzureCredential


In [3]:

# Connect mlClient
credential = DefaultAzureCredential()

ml_client = MLClient.from_config(
    credential=DefaultAzureCredential()
)


Found the config file in: /config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [4]:

ml_client = MLClient(
    credential=credential,
    subscription_id = ml_client.subscription_id,
    resource_group_name = ml_client.resource_group_name,
    workspace_name = ml_client.workspace_name
)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [5]:
env_name = "Pipeline_Flight-env"
exp_name = 'score_and_create_endpoint'
conda_file = "Environment.yml"
image_details ="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest"
DataStore_name = 'flightsdatastrore'
Blob_input = 'flightdelaydata.csv'

compute_instance_name = 'CPU432'

Prepare_data = "Prepare_date.py"
Train_data = 'Train_data.py'

In [6]:

# get data set
datastore = ml_client.datastores.get(DataStore_name)
print(datastore)
path = f"azureml://subscriptions/{ml_client.subscription_id}/resourcegroups/{ml_client.resource_group_name}/workspaces/{ml_client.workspace_name}/datastores/{datastore.name}/paths/{Blob_input}"


account_name: dp100n
container_name: flightcontainer
credentials: {}
endpoint: core.windows.net
id: /subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourceGroups/dp-100_trail/providers/Microsoft.MachineLearningServices/workspaces/dp-100/datastores/flightsdatastrore
name: flightsdatastrore
protocol: https
tags: {}
type: azure_blob



In [7]:

# Create environment
env = Environment(
    name = f"{env_name}",
    description="Custom AML environment",
    conda_file=f"{conda_file}",
    image = f"{image_details}" 
    # version = '1'
)

In [8]:


# Register environment
ml_client.environments.create_or_update(env)

# print("Environment created successfully")

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'Pipeline_Flight-env', 'description': 'Custom AML environment', 'tags': {}, 'properties': {'azureml.labels': 'latest'}, 'print_as_yaml': False, 'id': '/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourceGroups/dp-100_trail/providers/Microsoft.MachineLearningServices/workspaces/dp-100/environments/Pipeline_Flight-env/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpu432/code/Users/brenjoelsikha/PPipeline_Project', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x714f58ad4280>, 'serialize': <msrest.serialization.Serializer object at 0x714f58ad77f0>, 'version': '1', 'conda_file': {'channels': ['conda-forge'], 'dependencies': ['python=3

In [9]:




# # Register the original data set  URI FILE
# raw_data = Data(
#     path=path,
#     type=AssetTypes.URI_FILE,
#     name=DataAsset_Name,
#     # version="Main",
#     description="Raw flight delay dataset"
# )

# ml_client.data.create_or_update(raw_data)

In [10]:
# # Create a compute instance

# compute_instance_name = "test-my-compute-instance"
# compute_instance = ComputeInstance(
#     name=compute_instance_name ,
#     size="STANDARD_DS3_V2",
# )

# ml_client.begin_create_or_update(compute_instance).result()

# print("Compute instance created")

In [11]:

# # Create compute cluster 
# compute_cluster_name = 'testCluster'
# cpu_cluster = AmlCompute(
#     name=compute_cluster_name,
#     type="amlcompute",
#     size="STANDARD_DS2_V2",
#     min_instances=0,
#     max_instances=1,
#     idle_time_before_scale_down=120,
# )

# ml_client.begin_create_or_update(cpu_cluster).result()

# print("Compute cluster created")

In [12]:


# Step 1 - Prepare Data
prepare_data_step = command(
    name= 'prepare_data',
    display_name="Prepare Data",

    code="./src",

    command=f"""
    python {Prepare_data} \
    --input_data ${{{{inputs.input_data}}}} \
    --prepped_data ${{{{outputs.prepped_data}}}}
    """,

    inputs={
        "input_data": Input(
            type=AssetTypes.URI_FILE,
            path=path
        )
    },

    outputs={
        "prepped_data": Output(
            type=AssetTypes.URI_FOLDER
        )
    },

    environment=f'{env_name}@latest',
    compute=compute_instance_name
)

# Step 2 - Train Model
train_model_step = command(
    name='train_data',
    display_name="Train Model",

    code="./src",

    command=f"""
    python {Train_data} \
    --prepped_data ${{{{inputs.prepped_data}}}} \
    --model_output ${{{{outputs.model_output}}}}
    """,

    inputs={
        "prepped_data": Input(
            type=AssetTypes.URI_FOLDER
        ),
    },
    outputs = { 
        'model_output' : Output(
            type=AssetTypes.URI_FOLDER
            ),
        },
    environment=f'{env_name}@latest',  # "pipeline_env@latest",
    compute=compute_instance_name
)


In [13]:

# Build Pipeline
@pipeline()
def training_pipeline():

    # Step 1 execution
    prep_step = prepare_data_step()
    prep_step.compute=compute_instance_name

    # Step 2 execution with output from step 1
    train_step = train_model_step(
        prepped_data=prep_step.outputs.prepped_data
    )
    train_step.compute = compute_instance_name

    #step 3 execution


    return {
        "prepared_data": prep_step.outputs.prepped_data,
        "model_output": train_step.outputs.model_output
    }


In [14]:

# Create pipeline job
pipeline_job = training_pipeline()


# Experiment name
pipeline_job.experiment_name = exp_name

# Equivalent of regenerate_outputs=True
pipeline_job.settings.force_rerun = True



In [15]:
# Submit pipeline
returned_job = ml_client.jobs.create_or_update(pipeline_job)

ml_client.jobs.stream(returned_job.name)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
pathOnCompute is not a known attribute

RunId: careful_nutmeg_4vs23f1dth
Web View: https://ml.azure.com/runs/careful_nutmeg_4vs23f1dth?wsid=/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourcegroups/dp-100_trail/workspaces/dp-100

Streaming logs/azureml/executionlogs.txt

[2026-05-29 07:52:44Z] Submitting 1 runs, first five are: 9c62ab67:40113081-45d0-4768-84f4-bd4db2c830a0
[2026-05-29 07:54:17Z] Completing processing run id 40113081-45d0-4768-84f4-bd4db2c830a0.
[2026-05-29 07:54:18Z] Submitting 1 runs, first five are: 0ff3b4b5:55e75210-861d-41db-a780-d24bb54775dc
[2026-05-29 07:54:50Z] Completing processing run id 55e75210-861d-41db-a780-d24bb54775dc.

Execution Summary
RunId: careful_nutmeg_4vs23f1dth
Web View: https://ml.azure.com/runs/careful_nutmeg_4vs23f1dth?wsid=/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourcegroups/dp-100_trail/workspaces/dp-100



In [16]:
# # # Stop compute instance
# ml_client.compute.begin_stop(compute_instance_name).wait()

# print("Stopped")

In [17]:
# # # Delete compute instance
# ml_client.compute.begin_delete(compute_instance_name).wait()

# print("Compute instance deleted")

In [18]:
# Delete Compute Cluster 
# ml_client.compute.begin_delete(compute_cluster_name).wait()

In [19]:
# from azure.identity import InteractiveBrowserCredential

# auth_credential = InteractiveBrowserCredential()

# token = credential.get_token(
#     "https://ml.azure.com/.default"
# )

# access_token = token.token

In [20]:
# import requests
# import json

# exp_name = "pipeline_endpoint_exp"
# endpoint = "https://eastus2.api.azureml.ms/pipelines/v1.0/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourceGroups/dp-100_trail/providers/Microsoft.MachineLearningServices/workspaces/dp-100/PipelineRuns/PipelineEndpointSubmit/Id/6bb32fbc-2a8f-4c1e-8464-24bc50f00171"

# headers = {
#     "Authorization": f"Bearer {access_token}",
#     "Content-Type": "application/json"
# }

# payload = {
#     "ExperimentName": "PipelineTrigger"
# }

# response = requests.post(
#     endpoint,
#     headers=headers,
#     json=payload
# )

# print(response.status_code)
# print(response.text)

## 2


## 3

In [21]:
# # ---------------------------------------------------
# # CREATE PIPELINE JOB
# # ---------------------------------------------------

# pipeline_job = training_pipeline()

# pipeline_job.settings.force_rerun = True


# # ---------------------------------------------------
# # CREATE BATCH ENDPOINT
# # ---------------------------------------------------

# from azure.ai.ml.entities import (
#     BatchEndpoint,
#     PipelineComponentBatchDeployment
# )

# endpoint_name = "training-pipeline-endpoint"

# endpoint = BatchEndpoint(
#     name=endpoint_name,
#     description="Training pipeline endpoint"
# )

# ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

# print("Endpoint created")


# # ---------------------------------------------------
# # CREATE DEPLOYMENT
# # ---------------------------------------------------

# deployment = PipelineComponentBatchDeployment(
#     name="training-deployment",
#     endpoint_name=endpoint_name,

#     component=pipeline_job.component
# )

# ml_client.batch_deployments.begin_create_or_update(deployment).result()

# print("Deployment created")


# # ---------------------------------------------------
# # SET DEFAULT DEPLOYMENT
# # ---------------------------------------------------

# endpoint = ml_client.batch_endpoints.get(endpoint_name)

# endpoint.defaults.deployment_name = "training-deployment"

# ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

# print("Default deployment set")


# # ---------------------------------------------------
# # INVOKE ENDPOINT
# # ---------------------------------------------------

# job = ml_client.batch_endpoints.invoke(
#     endpoint_name=endpoint_name
# )

# print(job)
# ml_client.jobs.stream(
#     "pipelinejob-e71fd4a1-7ee1-4053-8afd-8a29f3a34f56"
# )

In [22]:
print(returned_job.name)

careful_nutmeg_4vs23f1dth


In [47]:

child_jobs = ml_client.jobs.list(parent_job_name=returned_job.name)

train_job_id = None

for job in child_jobs:
    if job.display_name == "train_step":
        train_job_id = job.name
        break

print("Train Job ID:", train_job_id)

model = Model(
    path=f"azureml://jobs/{train_job_id}/outputs/model_output/paths/model.pkl", #azureml/1276ed32-4680-4539-a2af-9a31aad4a2c1/model_output/
    # path=returned_job.outputs.model_output.path,
    name="flight-delay-model",
    type="custom_model"
)

registered_model = ml_client.models.create_or_update(model)




Train Job ID: 55e75210-861d-41db-a780-d24bb54775dc


In [44]:
print(returned_job.outputs)

{'prepared_data': <azure.ai.ml.entities._job.pipeline._io.base.PipelineOutput object at 0x714f584cae90>, 'model_output': <azure.ai.ml.entities._job.pipeline._io.base.PipelineOutput object at 0x714f584cb760>}


In [36]:
completed_job = ml_client.jobs.get(returned_job.name)

print(completed_job.outputs)
print(completed_job.outputs.model_output.path)

{'prepared_data': <azure.ai.ml.entities._job.pipeline._io.base.PipelineOutput object at 0x714efedc25c0>, 'model_output': <azure.ai.ml.entities._job.pipeline._io.base.PipelineOutput object at 0x714efedc00d0>}
None


40113081-45d0-4768-84f4-bd4db2c830a0
55e75210-861d-41db-a780-d24bb54775dc


In [40]:
child_jobs = ml_client.jobs.list(parent_job_name=returned_job.name)

for job in child_jobs:
    print(job.name, job.display_name)

40113081-45d0-4768-84f4-bd4db2c830a0 prep_step
55e75210-861d-41db-a780-d24bb54775dc train_step


In [45]:
print(registered_model.name)

flight-delay-model


## Create online endpoint


In [48]:
from azure.ai.ml.entities import ManagedOnlineEndpoint
import uuid

endpoint_name = "flight-endpoint-" + str(uuid.uuid4())[:8]

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    auth_mode="key"
)

ml_client.begin_create_or_update(endpoint).result()

print(endpoint_name)

HttpResponseError: (SubscriptionNotRegistered) Resource provider [N/A] isn't registered with Subscription [N/A]. Please see troubleshooting guide, available here: https://aka.ms/register-resource-provider
Code: SubscriptionNotRegistered
Message: Resource provider [N/A] isn't registered with Subscription [N/A]. Please see troubleshooting guide, available here: https://aka.ms/register-resource-provider

In [ ]:
from azure.ai.ml.entities import ManagedOnlineDeployment
from azure.ai.ml.entities import CodeConfiguration

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,

    model=registered_model,

    environment="flight-env@latest",

    code_configuration=CodeConfiguration(
        code="./",
        scoring_script="score.py"
    ),

    instance_type="Standard_DS3_v2",
    instance_count=1
)

ml_client.begin_create_or_update(deployment).result()